# Lab 12 — HDFS de um nó

> Extraído de `aulas/Aula_9_hadoop.ipynb` (seção 6, exercício 3, fechamento e resumo). As referências a "aula N" e a números de seção usam a numeração do notebook original.

## Preparação

Este notebook roda sozinho: as células abaixo recriam o que ele usa do notebook 01 (sessão Spark, `corpus.txt`, `mapper.py`, `reducer.py` e `orders.csv`).

In [ ]:
# !pip install -q pyspark==4.0.1 pyarrow

In [ ]:
import os, re, subprocess, shutil, time, random, glob
from collections import defaultdict
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Aula 9 - Hadoop")
    .master("local[*]")
    # spark.hadoop.* vai direto para a configuração do Hadoop. Aqui: blocos de 4 MB no disco local
    # (o padrão é 32 MB), para vermos vários blocos com um arquivo pequeno (seção 3)
    .config("spark.hadoop.fs.local.block.size", str(4 * 2**20))
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

BASE = "/tmp/aula9"
os.makedirs(BASE, exist_ok=True)

def sh(cmd, check=True):
    """Roda um comando de shell e mostra a saída (equivalente a ! no notebook)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"comando falhou ({r.returncode}): {cmd}")

print("Spark            :", spark.version)
print("Hadoop (cliente) :", spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

In [ ]:
jvm  = spark._jvm
conf = spark._jsc.hadoopConfiguration()

for scheme in ["file", "hdfs", "webhdfs", "s3a", "gs", "abfs"]:
    try:
        impl = jvm.org.apache.hadoop.fs.FileSystem.getFileSystemClass(scheme, conf).getName()
        print(f"{scheme + '://':<11} -> {impl}")
    except Exception:
        print(f"{scheme + '://':<11} -> (conector não instalado)")

In [ ]:
VOCAB = ("dados spark hadoop hdfs yarn mapreduce cluster bloco arquivo tabela python pandas "
         "parquet delta streaming janela modelo pipeline shuffle particao memoria disco rede "
         "driver executor task stage job lake bronze silver gold").split()
weights = [1 / (i + 1) for i in range(len(VOCAB))]          # distribuição de Zipf: poucas palavras muito frequentes

rng = random.Random(42)
CORPUS = f"{BASE}/corpus.txt"
with open(CORPUS, "w") as f:
    for _ in range(600_000):
        f.write(" ".join(rng.choices(VOCAB, weights, k=12)) + "\n")

path   = jvm.org.apache.hadoop.fs.Path(f"file://{CORPUS}")
fs     = path.getFileSystem(conf)
status = fs.getFileStatus(path)

print(f"Tamanho          : {status.getLen() / 1e6:.1f} MB")
print(f"Tamanho de bloco : {status.getBlockSize() / 2**20:.0f} MB  (definido no setup)")
print(f"Replicação       : {status.getReplication()}")
print(f"Blocos lógicos   : {-(-status.getLen() // status.getBlockSize())}")

In [ ]:
MAPPER = r'''
import sys, re
for line in sys.stdin:
    for word in re.findall(r"[a-z]+", line.lower()):
        print(f"{word}\t1")
'''

REDUCER = r'''
import sys
current, count = None, 0
for line in sys.stdin:
    word, n = line.rstrip("\n").split("\t")
    if word != current:              # chave nova: a anterior terminou (a entrada está ordenada)
        if current is not None:
            print(f"{current}\t{count}")
        current, count = word, 0
    count += int(n)
if current is not None:
    print(f"{current}\t{count}")
'''

with open(f"{BASE}/mapper.py", "w") as f:  f.write(MAPPER)
with open(f"{BASE}/reducer.py", "w") as f: f.write(REDUCER)

print("Saída do mapper (início):")
sh(f"head -n 1 {CORPUS} | python3 {BASE}/mapper.py")

In [ ]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# MapReduce com scripts: calcule a receita total (price * quantity) por região a partir do CSV abaixo,
# escrevendo um mapper e um reducer no estilo Hadoop Streaming e rodando com
#   cat | mapper | sort | reducer
# Confira o resultado com Spark ou Pandas.
# Dica: o mapper deve pular o cabeçalho e emitir "regiao<TAB>receita".

ORDERS_CSV = f"{BASE}/orders.csv"
rng = random.Random(7)
with open(ORDERS_CSV, "w") as f:
    f.write("order_id,region,price,quantity\n")
    for i in range(200_000):
        f.write(f"{i},{rng.choice(['North','Northeast','Midwest','Southeast','South'])},"
                f"{rng.uniform(10, 500):.2f},{rng.randint(1, 5)}\n")

# Sua solução:

---
## 6. Mão na massa: HDFS de um nó

Agora vamos instalar o Hadoop e subir um HDFS em **modo pseudo-distribuído**: NameNode e DataNode rodando na mesma máquina, como processos separados. É o mesmo software de um cluster de 1.000 nós, só que com um nó.

> Esta seção foi pensada para Colab ou Linux com Java instalado. O download tem cerca de 500 MB e leva alguns minutos. Se o link estiver fora do ar, veja as versões em https://archive.apache.org/dist/hadoop/common/

In [ ]:
HADOOP_VERSION = "3.4.1"          # mesma versão do cliente Hadoop embutido no PySpark 4.0
HADOOP_HOME    = f"/opt/hadoop-{HADOOP_VERSION}"
MIRROR = f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}"
# versão "lean" (sem o SDK da AWS, ~metade do tamanho); se não existir, baixa a completa
URLS = [f"{MIRROR}/hadoop-{HADOOP_VERSION}-lean.tar.gz", f"{MIRROR}/hadoop-{HADOOP_VERSION}.tar.gz"]

if not os.path.exists(HADOOP_HOME):
    sh(" || ".join(f"wget -q {u} -O /tmp/hadoop.tar.gz" for u in URLS))
    sh("tar -xzf /tmp/hadoop.tar.gz -C /opt")
    os.remove("/tmp/hadoop.tar.gz")

JAVA_HOME = os.path.dirname(os.path.dirname(os.path.realpath(shutil.which("java"))))
os.environ.update({
    "JAVA_HOME":       JAVA_HOME,
    "HADOOP_HOME":     HADOOP_HOME,
    "HADOOP_CONF_DIR": f"{HADOOP_HOME}/etc/hadoop",
    "PATH":            f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"],
    # o Colab roda como root; o Hadoop exige que isso seja declarado
    "HDFS_NAMENODE_USER": "root", "HDFS_DATANODE_USER": "root", "HDFS_SECONDARYNAMENODE_USER": "root",
})
sh("hadoop version | head -n 1")
print("JAVA_HOME:", JAVA_HOME)

### Configuração

Dois arquivos XML definem o cluster:

| Arquivo | Propriedade | Valor | Significado |
|---------|-------------|-------|-------------|
| `core-site.xml` | `fs.defaultFS` | `hdfs://localhost:9000` | Endereço do NameNode |
| `hdfs-site.xml` | `dfs.replication` | `1` | Só temos um DataNode |
| `hdfs-site.xml` | `dfs.namenode.name.dir` | pasta local | Onde o NameNode guarda os metadados |
| `hdfs-site.xml` | `dfs.datanode.data.dir` | pasta local | Onde o DataNode guarda os blocos |

In [ ]:
HDFS_DATA = "/tmp/hdfs-data"

def write_site(name, props):
    body = "\n".join(f"  <property><name>{k}</name><value>{v}</value></property>" for k, v in props.items())
    with open(f"{HADOOP_HOME}/etc/hadoop/{name}", "w") as f:
        f.write(f"<?xml version=\"1.0\"?>\n<configuration>\n{body}\n</configuration>\n")

write_site("core-site.xml", {"fs.defaultFS": "hdfs://localhost:9000"})
write_site("hdfs-site.xml", {
    "dfs.replication":         1,
    "dfs.namenode.name.dir":   f"file://{HDFS_DATA}/namenode",
    "dfs.datanode.data.dir":   f"file://{HDFS_DATA}/datanode",
    "dfs.permissions.enabled": "false",
})

# Java 17+ (exigido pelo PySpark 4) precisa liberar alguns módulos internos para o Hadoop
with open(f"{HADOOP_HOME}/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write(f"\nexport JAVA_HOME={JAVA_HOME}\n")
    f.write('export HADOOP_OPTS="$HADOOP_OPTS --add-opens=java.base/java.lang=ALL-UNNAMED '
            '--add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED"\n')

sh(f"cat {HADOOP_HOME}/etc/hadoop/hdfs-site.xml")

In [ ]:
# Formatar o NameNode (só na primeira vez: apaga os metadados!) e subir os dois processos
shutil.rmtree(HDFS_DATA, ignore_errors=True)
sh("hdfs namenode -format -force -nonInteractive 2>&1 | grep -i 'successfully formatted'", check=False)

sh("hdfs --daemon start namenode")
sh("hdfs --daemon start datanode")
sh("hdfs dfsadmin -safemode wait")     # o NameNode inicia em modo somente leitura até os DataNodes se registrarem

sh("jps", check=False)                 # processos Java rodando
sh("hdfs dfsadmin -report | head -n 12")

`jps` deve listar `NameNode` e `DataNode`. No Linux local, a interface web do NameNode fica em http://localhost:9870 (no Colab ela não é acessível diretamente).

### 6.1 Comandos `hdfs dfs`

Os comandos imitam os do Linux, com `hdfs dfs -` na frente:

| Linux | HDFS |
|-------|------|
| `ls -lh` | `hdfs dfs -ls -h` |
| `mkdir -p` | `hdfs dfs -mkdir -p` |
| `cp local destino` | `hdfs dfs -put local destino` |
| `cp destino local` | `hdfs dfs -get destino local` |
| `cat` / `tail` | `hdfs dfs -cat` / `hdfs dfs -tail` |
| `du -sh` | `hdfs dfs -du -s -h` |
| `rm -r` | `hdfs dfs -rm -r` |

In [ ]:
sh("hdfs dfs -mkdir -p /aula9/raw")
sh(f"hdfs dfs -put -f {CORPUS} /aula9/raw/corpus.txt")

# o tamanho de bloco é definido por arquivo, na escrita: aqui, 4 MB para gerar vários blocos
sh(f"hdfs dfs -D dfs.blocksize={4 * 2**20} -put -f {CORPUS} /aula9/raw/corpus_4mb.txt")

sh("hdfs dfs -ls -h /aula9/raw")
sh("hdfs dfs -du -s -h /aula9")
sh("hdfs dfs -cat /aula9/raw/corpus.txt | head -n 2")

In [ ]:
# fsck mostra os blocos de um arquivo e em quais DataNodes estão
sh("hdfs fsck /aula9/raw/corpus_4mb.txt -files -blocks -locations | grep -E 'blk_|Total blocks|corpus' | head -n 8")

In [ ]:
# Os blocos são só arquivos comuns no disco do DataNode
sh(f"find {HDFS_DATA}/datanode -name 'blk_*' -not -name '*.meta' | head -n 5")
sh(f"find {HDFS_DATA}/datanode -name 'blk_*' -not -name '*.meta' | wc -l")

Cada `blk_<id>` é um pedaço de até 4 MB do arquivo, e o `.meta` ao lado guarda os checksums usados para detectar corrupção. O NameNode só sabe **quais blocos formam cada arquivo e onde estão**; o conteúdo nunca passa por ele.

### 6.2 Spark lendo e escrevendo no HDFS

Para o Spark, o HDFS é só mais um esquema de URL. O código é o mesmo das aulas anteriores.

In [ ]:
HDFS = "hdfs://localhost:9000"

print("Partições (textFile, 1 por bloco):", sc.textFile(f"{HDFS}/aula9/raw/corpus_4mb.txt").getNumPartitions())

words = (
    spark.read.text(f"{HDFS}/aula9/raw/corpus.txt")
    .select(F.explode(F.split("value", " ")).alias("word"))
    .groupBy("word").count()
)
words.write.mode("overwrite").parquet(f"{HDFS}/aula9/gold/wordcount")

sh("hdfs dfs -ls -h /aula9/gold/wordcount")
spark.read.parquet(f"{HDFS}/aula9/gold/wordcount").orderBy(F.desc("count")).show(5)

### 6.3 Um job MapReduce de verdade

Os mesmos `mapper.py` e `reducer.py` da seção 4.2, agora executados pelo Hadoop Streaming, lendo e escrevendo no HDFS. Sem YARN configurado, o Hadoop usa o executor local (`mapreduce.framework.name=local`), mas o fluxo é idêntico ao de um cluster.

In [ ]:
streaming_jar = glob.glob(f"{HADOOP_HOME}/share/hadoop/tools/lib/hadoop-streaming-*.jar")[0]
sh("hdfs dfs -rm -r -f /aula9/out/wordcount_mr")

t0 = time.perf_counter()
sh(f"""hadoop jar {streaming_jar} \
      -files {BASE}/mapper.py,{BASE}/reducer.py \
      -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
      -input /aula9/raw/corpus_4mb.txt -output /aula9/out/wordcount_mr 2>&1 | grep -E 'map 100%|completed successfully|Launched map tasks'""",
   check=False)
print(f"\nTempo do job MapReduce: {time.perf_counter() - t0:.1f} s")

sh("hdfs dfs -ls /aula9/out/wordcount_mr")
sh("hdfs dfs -cat /aula9/out/wordcount_mr/part-* | sort -k2 -nr | head -n 5")

O arquivo `_SUCCESS` é um marcador: o job terminou e a saída está completa. Leitores verificam esse arquivo antes de consumir a pasta. O Spark escreve o mesmo marcador, herança direta do Hadoop.

### 6.4 Pandas e PyArrow lendo o HDFS

Pandas não fala HDFS sozinho. Há dois caminhos:

| Caminho | Como funciona | Prós e contras |
|---------|---------------|----------------|
| **PyArrow `HadoopFileSystem`** | Usa a biblioteca nativa `libhdfs` (que inicia uma JVM dentro do Python) | Rápido, lê Parquet direto; exige Hadoop instalado e `CLASSPATH` configurado |
| **WebHDFS (REST)** | Requisições HTTP ao NameNode (porta 9870), que redireciona para o DataNode | Só precisa de `requests`; mais lento para arquivos grandes |

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs as pafs

# libhdfs precisa encontrar as bibliotecas Java do Hadoop
os.environ["CLASSPATH"] = subprocess.run(["hadoop", "classpath", "--glob"],
                                         capture_output=True, text=True, check=True).stdout.strip()
os.environ["ARROW_LIBHDFS_DIR"] = f"{HADOOP_HOME}/lib/native"

hdfs = pafs.HadoopFileSystem("localhost", 9000)

for info in hdfs.get_file_info(pafs.FileSelector("/aula9", recursive=False)):
    print(f"{info.type.name:<10} {info.path}")

# Parquet do HDFS -> Arrow -> Pandas
wc = pq.read_table("/aula9/gold/wordcount", filesystem=hdfs).to_pandas()
print()
print(wc.sort_values("count", ascending=False).head())

In [ ]:
# Pandas -> HDFS: escrevendo um Parquet pelo mesmo filesystem
summary = wc.assign(length=wc["word"].str.len()).groupby("length", as_index=False)["count"].sum()
pq.write_table(pa.Table.from_pandas(summary), "/aula9/gold/count_by_length.parquet", filesystem=hdfs)
sh("hdfs dfs -ls -h /aula9/gold")

In [ ]:
# Alternativa sem libhdfs: WebHDFS, a API REST do HDFS
import io, requests

WEBHDFS = "http://localhost:9870/webhdfs/v1"

listing = requests.get(f"{WEBHDFS}/aula9/raw?op=LISTSTATUS").json()
for st in listing["FileStatuses"]["FileStatus"]:
    print(f"{st['pathSuffix']:<16} {st['length'] / 1e6:6.1f} MB  bloco={st['blockSize'] / 2**20:.0f} MB  réplicas={st['replication']}")

# OPEN: o NameNode responde com um redirect para o DataNode que tem o bloco (requests segue sozinho)
r = requests.get(f"{WEBHDFS}/aula9/out/wordcount_mr/part-00000?op=OPEN")
pd.read_csv(io.BytesIO(r.content), sep="\t", names=["word", "count"]).nlargest(3, "count")

Repare no redirect do `OPEN`: é a arquitetura do HDFS em ação. O cliente pergunta ao NameNode **onde** está o dado e lê **diretamente do DataNode**.

### Exercício

Faça este exercício antes de desligar o HDFS na seção 6.5.

In [ ]:
# ── Exercício 3 (requer o HDFS da seção 6) ──────────────────────────────────
# a) Envie o orders.csv do exercício 1 para o HDFS com blocos de 1 MB.
# b) Use fsck para descobrir quantos blocos ele tem.
# c) Leia o arquivo com sc.textFile e com spark.read.csv. Quantas partições cada um gera? Por quê?

# Sua solução:

In [ ]:
# ── Solução 3 ───────────────────────────────────────────────────────────────
sh(f"hdfs dfs -D dfs.blocksize={2**20} -put -f {ORDERS_CSV} /aula9/raw/orders.csv")
sh("hdfs fsck /aula9/raw/orders.csv -files -blocks | grep 'Total blocks'")

print("sc.textFile   :", sc.textFile(f"{HDFS}/aula9/raw/orders.csv").getNumPartitions(), "partições")
print("spark.read.csv:", spark.read.csv(f"{HDFS}/aula9/raw/orders.csv", header=True).rdd.getNumPartitions(), "partições")
# textFile usa o InputFormat do Hadoop: uma partição por bloco (split).
# spark.read divide pelo próprio critério (maxPartitionBytes e número de núcleos), não pelos blocos.

### 6.5 Desligando

In [ ]:
sh("hdfs --daemon stop datanode")
sh("hdfs --daemon stop namenode")
sh("jps", check=False)

---
## 8. Fechando o curso

```
                     ┌──────────────────────────────────────────────────────────┐
  Consumo            │ BI, dashboards, APIs, modelos                    aula 7   │
                     ├──────────────────────────────────────────────────────────┤
  Processamento      │ Spark DataFrame / SQL                            aula 3   │
                     │ performance: shuffles, joins, skew, AQE          aula 4   │
                     │ streaming                                        aula 6   │
                     │ uma máquina basta? Pandas, Polars, DuckDB        aula 8   │
                     ├──────────────────────────────────────────────────────────┤
  Tabelas            │ Medallion + Delta Lake (ACID, MERGE, time travel) aula 5  │
                     ├──────────────────────────────────────────────────────────┤
  Recursos           │ local[*] | YARN | Kubernetes                     aula 9   │
                     ├──────────────────────────────────────────────────────────┤
  Armazenamento      │ Hadoop FileSystem API -> disco | HDFS | S3/GCS   aula 9   │
                     └──────────────────────────────────────────────────────────┘
```

---
## Resumo

| Conceito | Hadoop | No Spark |
|----------|--------|----------|
| Arquivos distribuídos | HDFS: blocos de 128 MB, 3 réplicas | `hdfs://` via API FileSystem |
| Metadados | NameNode (memória) | — (catálogos: Hive Metastore, Unity) |
| Processamento | MapReduce: map, shuffle e sort, reduce | Stages separados por shuffles |
| Recursos | YARN: ResourceManager, containers | `--master yarn`, executors = containers |
| Mover código até os dados | Localidade de blocos | `Locality Level` na Spark UI |
| Saída concluída | Marcador `_SUCCESS` | O mesmo marcador |
| Configurações | `core-site.xml`, `hdfs-site.xml` | `spark.hadoop.*` |
| Nuvem | Conectores S3A, GCS, ABFS | `s3a://`, `gs://`, `abfs://` |

| Comando | Para quê |
|---------|----------|
| `hdfs dfs -ls / -put / -get / -cat / -du / -rm` | Manipular arquivos |
| `hdfs fsck <caminho> -files -blocks -locations` | Ver blocos e réplicas |
| `hdfs dfsadmin -report` | Estado do cluster |
| `hadoop jar hadoop-streaming.jar -mapper ... -reducer ...` | MapReduce com scripts |
| `spark-submit --master yarn --deploy-mode cluster` | Rodar Spark no cluster Hadoop |

In [ ]:
spark.stop()